In [1]:
!pip install flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [flask]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!-h

/usr/bin/sh: line 1: free: command not found


In [3]:
!grep -E "MemTotal|MemAvailable|SwapTotal|SwapFree" /proc/meminfo

MemTotal:        8201236 kB
MemAvailable:    6653252 kB
SwapTotal:             0 kB
SwapFree:              0 kB


In [4]:
%%writefile chip_bench.py
import sys, time
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(0)
REPS = int(sys.argv[1]) if len(sys.argv) > 1 else 20

model = tf.keras.Sequential([
    tf.keras.Input(shape=(512,)),
    tf.keras.layers.Dense(1024, activation="relu"),
    tf.keras.layers.Dense(1024, activation="relu"),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

@tf.function
def score(x):
    return model(x, training=False)

x = tf.constant(np.random.RandomState(0).rand(8192, 512).astype("float32"))
out = score(x).numpy()  # warm-up

t = time.time()
for _ in range(REPS):
    out = score(x).numpy()
dt = time.time() - t
print(f"pairs scored per second: {int(8192 * REPS / dt)}")
print(f"seconds total: {dt:.2f}")
print("first 3 scores:", out[:3, 0])

Writing chip_bench.py


In [5]:
import sys
!NNPA_DEVICES=0 {sys.executable} chip_bench.py

pairs scored per second: 41918
seconds total: 3.91
first 3 scores: [0.41312236 0.44746727 0.4380148 ]


In [6]:
!{sys.executable} chip_bench.py

pairs scored per second: 237224
seconds total: 0.69
first 3 scores: [0.41259766 0.44726562 0.43847656]


In [7]:
!ZDNN_ENABLE_PRECHECK=true TF_CPP_MIN_LOG_LEVEL=0 TF_CPP_VMODULE='remapper=1' {sys.executable} chip_bench.py 2 2>&1 | head -40

2026-10-17 19:19:33.921975: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type CPU is enabled.
2026-10-17 19:19:33.926785: I tensorflow_plugin/grappler/optimizers/remapper.cc:1212] ZRemapper checking node 0 of 22: x (_Arg)
2026-10-17 19:19:33.926797: I tensorflow_plugin/grappler/optimizers/remapper.cc:1212] ZRemapper checking node 1 of 22: sequential_1_dense_1_biasadd_readvariableop_resource (_Arg)
2026-10-17 19:19:33.926800: I tensorflow_plugin/grappler/optimizers/remapper.cc:1212] ZRemapper checking node 2 of 22: sequential_1/dense_1/BiasAdd/ReadVariableOp (ReadVariableOp)
2026-10-17 19:19:33.926811: I tensorflow_plugin/grappler/optimizers/remapper.cc:1269] ZRemapper !IsZCandidate
2026-10-17 19:19:33.926815: I tensorflow_plugin/grappler/optimizers/remapper.cc:1098] ZRemapper MaybeUnstickifyInput sequential_1/dense_1/BiasAdd/ReadVariableOp (ReadVariableOp) input: 0
2026-10-17 19:19:33.926831: I tensorflow_plugin/grappler/opti

In [8]:
!ZDNN_ENABLE_PRECHECK=true TF_CPP_MIN_LOG_LEVEL=0 TF_CPP_VMODULE='remapper=1' {sys.executable} chip_bench.py 2 2>&1 | grep -E "remapping to Z op|ZStickify" | head -20

2026-10-17 19:22:41.574192: I tensorflow_plugin/grappler/optimizers/remapper.cc:1417] ZRemapper remapping to Z op
2026-10-17 19:22:41.574204: I tensorflow_plugin/grappler/optimizers/remapper.cc:1008] ZRemapper add ZStickify node x_out_0_ZDNN_NHWC
2026-10-17 19:22:41.574314: I tensorflow_plugin/grappler/optimizers/remapper.cc:1417] ZRemapper remapping to Z op
2026-10-17 19:22:41.574401: I tensorflow_plugin/grappler/optimizers/remapper.cc:1417] ZRemapper remapping to Z op


In [9]:
!pwd

/home/ibm-user/shared


In [10]:
import json
json.dump({
  "chip_on":  {"pairs_per_sec": 237224, "seconds": 0.69},
  "chip_off": {"pairs_per_sec": 41918,  "seconds": 3.91},
  "note": "Same model and data, accelerator on vs off. 4-layer neural network, 8,192 pairs per batch, 20 repeats, 3 virtual CPUs."
}, open("benchmark.json", "w"), indent=2)

In [11]:
import os
os.makedirs("docs", exist_ok=True)
notes = """# LinuxONE AI accelerator results

Test: chip_bench.py (4-layer neural network, 8,192 pairs per batch, 20 repeats)
Same model and data both times. Only difference: accelerator on or off.

| Setting | Pairs scored per second | Total time |
|---|---|---|
| Accelerator OFF (NNPA_DEVICES=0) | 41,918 | 3.91 s |
| Accelerator ON (default) | 237,224 | 0.69 s |

Speedup: about 5.7x.
Scores differ by about 0.0005 between runs (the accelerator uses lower precision internally).

Evidence: with ZDNN_ENABLE_PRECHECK=true, TF_CPP_MIN_LOG_LEVEL=0 and TF_CPP_VMODULE='remapper=1',
the log shows "ZRemapper remapping to Z op" for 3 layers and a ZStickify node.
Screenshot: docs/chip_proof.png (upload it here).

Environment: LinuxONE community cloud VM, 3 virtual CPUs, 8 GB RAM, z/VM 7.3.0,
lscpu machine type 9175, CPU flags include nnpa and vxe2,
TensorFlow 2.18.1 with ibm-zdnn-plugin 2.3.0, libzdnn.so.0 loaded.

Caveats: small model with random weights, one test. Rerun with the real match scorer and update this file.

Reproduce:
  NNPA_DEVICES=0 python chip_bench.py
  python chip_bench.py
"""
open("docs/chip_results.md", "w").write(notes)
print("saved")

saved
